# Run inference and evaluation on Non-standard Speech data in Kenyan Languages

* Datasets are hosted on CDLI's HuggingFace page
    * English: 
        * https://huggingface.co/datasets/cdli/kenyan_english_nonstandard_speech_v0
        * https://huggingface.co/datasets/cdli/kenyan_english_nonstandard_speech_v0.9
    * Swahili: 
        * https://huggingface.co/datasets/cdli/kenyan_swahili_nonstandard_speech_v0
        * https://huggingface.co/datasets/cdli/kenyan_swahili_nonstandard_speech_v0.9
* until full public release, you need to request access to use them

In [1]:
# need to login to Hugging Face Hub to access the datasets
# use your huggingface token (you can get that under "Access Tokens" in your HuggingFace account)

from huggingface_hub import login
HF_TOKEN = input()
login(token=HF_TOKEN)

# Imports and defs

In [ ]:
!nvidia-smi

In [2]:
import datasets
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
import os

import torch
from tqdm.auto import tqdm
from transformers.pipelines.pt_utils import Dataset

In [3]:
# more efficient dataset handling
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

torch.set_num_threads(1)
torch.get_num_threads()

cache: False


1

In [4]:
# note: this is slower when more stored in dataset (eg features), as it leads to data being copied around
class AudioTextDataset(Dataset):

    DEFAULT_UTTERANCE_FIELDS = ['audio_id', 'speaker_id',  'language', 
                         'prompt_type', 'prompt_id',
                         'transcription', 'audio_length', 'transcript_length']

    def __init__(self, dataset: Dataset, utterance_fields=DEFAULT_UTTERANCE_FIELDS):
        self.dataset = dataset
        self.utterance_fields = utterance_fields

    def __len__(self):
        return len(self.dataset)


    def __getitem__(self, i):

        result = {k: self.dataset[i][k] for k in self.utterance_fields}
        # these two fields are required for the ASR pipeline
        result['sampling_rate'] = self.dataset[i]['audio']['sampling_rate']
        result['raw'] = self.dataset[i]['audio']['array']
        return result


In [5]:
from evaluate import load as metrics_loader
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

wer_metric = metrics_loader("wer")
cer_metric = metrics_loader("cer")

transcript_normalizer = BasicTextNormalizer()

def get_wer_cer(references, predictions,
                calculate_utterance_level_averaged_wer=False,
                normalize=True, verbose=True,
                ):
  # calculate_utterance_level_averaged_wer -- we first calculate the WER per
  # utterance and then average. This is not the standard way to calculate WER
  # on a corpus, but in a scenario of high WER (as for NSS) this allows to cap
  # at 1.0 on a per-utterance level.
  rs = references
  ps = predictions
  if normalize:
    pred_strs = [transcript_normalizer(x) for x in predictions]
    label_strs = [transcript_normalizer(x) for x in references]
  if calculate_utterance_level_averaged_wer:
    wers = []
    cers = []
    for pred_str, label_str in zip(pred_strs, label_strs):
      p = transcript_normalizer(pred_str)
      l = transcript_normalizer(label_str)
      wer = wer_metric.compute(predictions=[p], references=[l])
      cer = cer_metric.compute(predictions=[p], references=[l])
      wers.append(wer)
      cers.append(cer)
      if verbose:
        print(label_str, '-->', pred_str, '-->', wer, cer)
    wer = np.mean([min(1.0, x) for x in wers])
    cer = np.mean([min(1.0, x) for x in cers])
  else:
    wer =  min(1, wer_metric.compute(references=label_strs, predictions=pred_strs))
    cer =  min(1, cer_metric.compute(references=label_strs, predictions=pred_strs))

    if verbose:
      for pred_str, label_str in zip(pred_strs, label_strs):
        print(label_str, '-->', pred_str)

  return (wer, cer)

In [6]:
def prepare_results(results):
    df = pd.DataFrame(results)
    def clean_res(row):
        for f in df.columns:
            v = row[f]
            if isinstance(v, list):
                row[f] = v[0]
        return row
    df = df.apply(clean_res, axis=1)
    df['prediction'] = df['text'].str.strip() if isinstance(df['text'], str) else df['text']
   
    df['ground_truth'] = df['transcription'].str.strip() if isinstance(df['transcription'], str) else df['transcription']
    df = df.drop(columns=['text', 'transcription'])
    return df

def add_speaker_metadata(results_df, metadata_df):
    # Merge metadata with results_df on 'speaker_id'
    merged_df = results_df.merge(metadata_df, on='speaker_id', how='left')

    # Check if any speaker_id in results_df is missing in metadata_df
    missing_speakers = set(results_df['speaker_id']) - set(metadata_df['speaker_id'])
    if missing_speakers:
        print(f"Warning: Missing metadata for speakers: {missing_speakers}")

    return merged_df

def calculate_error_rates(results_df, verbose=False):

  def get_row_wer(row):
    reference = transcript_normalizer(row['ground_truth'])
    prediction = transcript_normalizer(row['prediction'])
    return get_wer_cer(references=[reference], predictions=[prediction], normalize=True, verbose=verbose)[0]

  def get_row_cer(row):
    reference = transcript_normalizer(row['ground_truth'])
    prediction = transcript_normalizer(row['prediction'])
    return get_wer_cer(references=[reference], predictions=[prediction], normalize=True, verbose=verbose)[1]




  # add normalized ground truth and prediction
  results_df['ground_truth_normalized'] = results_df['ground_truth'].apply(lambda x: transcript_normalizer(x))
  results_df['prediction_normalized'] = results_df['prediction'].apply(lambda x: transcript_normalizer(x))

  results_df['wer'] = results_df.apply(get_row_wer, axis=1)
  results_df['cer'] = results_df.apply(get_row_cer, axis=1)

  overall_wer_cer = get_wer_cer(references=results_df.ground_truth.tolist(),
                                predictions=results_df.prediction.tolist(),
                                calculate_utterance_level_averaged_wer=False,
                                normalize=True, verbose=verbose)

  avg_utterance_level_wer_cer = get_wer_cer(references=results_df.ground_truth.tolist(),
                                predictions=results_df.prediction.tolist(),
                                calculate_utterance_level_averaged_wer=True,
                                normalize=True, verbose=verbose)


  print('Overall WER (normalized):', round(overall_wer_cer[0],3))
  print('Overall CER (normalized):', round(overall_wer_cer[1],3))
  print('Avg WER (normalized):', round(avg_utterance_level_wer_cer[0],3))
  print('Avg CER (normalized):', round(avg_utterance_level_wer_cer[1],3))

  return results_df

In [7]:
def load_dataset(dataset_name, split='test', limit_to_30_seconds=True):
    """
    Load a dataset from Hugging Face Hub.
    If limit_to_30_seconds is True, will only load examples with audio length <= 30 seconds.
    """
    if split not in ['train', 'test', 'validation']:
        raise ValueError("split must be one of 'train', 'test', or 'validation'")
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda example: example['audio_length'] <= 30)
        print(f"Filtered dataset from {orig_len} to {len(ds)} examples with audio length <= 30 seconds")
    return ds

def load_speaker_metadata(dataset_name, metadata_filename="speaker_metadata.csv"):
    """
    Load speaker metadata from a dataset.
    """
    metadata_file_path = hf_hub_download(
        repo_id=dataset_name,
        filename=metadata_filename,
        repo_type="dataset"
    )
    # print(f"File downloaded to: {metadata_file_path}")
    seperator = '\t' if metadata_filename.endswith('.tsv') else ','
    metadata_df = pd.read_csv(metadata_file_path, sep=seperator)
    # map severity
    if 'severity_speech_impairment' in metadata_df.columns:
        metadata_df['severity'] = metadata_df['severity_speech_impairment'].apply(lambda x: x.split()[0].lower())
        fields_to_remove = ['severity_speech_impairment']
        metadata_df = metadata_df.drop(columns=fields_to_remove)

    return metadata_df

In [8]:
# because the kernel is running remotely, the file is stored remotely as well
# this code helps to download it

from IPython.display import HTML
import base64

def create_download_link(filename):
    with open(filename, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    
    html = f'<a href="data:application/octet-stream;base64,{data}" download="{filename}">📥 Download {filename}</a>'
    display(HTML(html))

# # Usage
# create_download_link(filename)

# Configs

In [ ]:
WHISPER_MODEL_NAME = "openai/whisper-tiny"
# WHISPER_MODEL_NAME = "openai/whisper-small"
# WHISPER_MODEL_NAME = "openai/whisper-large-v3"

# # whisper tuned with extra Swahili data from common voice (standard speech)
# WHISPER_MODEL_NAME = "cdli/whisper-large-v3-Swahili_finetuned_small_CV20"
# WHISPER_MODEL_NAME = "cdli/whisper-small-Swahili_finetuned_small_CV20"

# models tuned for non-standard speech
# WHISPER_MODEL_NAME = "cdli/whisper-large-v3_finetuned_kenyan_english_nonstandard_speech_v0.9"
# WHISPER_MODEL_NAME = "cdli/whisper-small_finetuned_kenyan_english_nonstandard_speech_v0.9"
# WHISPER_MODEL_NAME = "cdli/whisper-tiny_finetuned_kenyan_english_nonstandard_speech_v0.9"

# V0 has CSV metadata, V0.9 has TSV metadata
# METADATA_FILENAME = "speaker_metadata.csv"
METADATA_FILENAME = "speaker_metadata.tsv"


# DATASET_NAME = "cdli/kenyan_english_nonstandard_speech_v0.9"
# LANGUAGE = "en"

DATASET_NAME = "cdli/kenyan_swahili_nonstandard_speech_v0.9"
LANGUAGE = "sw"

# DATASET_NAME = "cdli/common_voice_swahili_small"
# LANGUAGE = "sw"




# Load datasets

In [10]:
metadata_df = load_speaker_metadata(DATASET_NAME, metadata_filename=METADATA_FILENAME)

In [11]:
test_dataset = load_dataset(DATASET_NAME, split='test', limit_to_30_seconds=True)
print(f"Loaded TEST dataset with {len(test_dataset)} examples")

Filter:   0%|          | 0/865 [00:00<?, ? examples/s]

Filtered dataset from 865 to 554 examples with audio length <= 30 seconds
Loaded TEST dataset with 554 examples


In [12]:
dev_dataset = load_dataset(DATASET_NAME, split='validation', limit_to_30_seconds=True)
print(f"Loaded DEV dataset with {len(dev_dataset)} examples")

Filter:   0%|          | 0/417 [00:00<?, ? examples/s]

Filtered dataset from 417 to 272 examples with audio length <= 30 seconds
Loaded DEV dataset with 272 examples


# Load model and pipeline

In [13]:
from transformers import pipeline

# Note: if handling data with more than 30 seconds, you need to set return_timestamps=True
# That will require batch size of 1 and also slow down the inference significantly due to timestamping

print("Loading model:", WHISPER_MODEL_NAME)
# Create pipeline
pipe = pipeline("automatic-speech-recognition", 
                model=WHISPER_MODEL_NAME,
                )


Loading model: openai/whisper-tiny


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


# Run inference on dataset

In [15]:
generate_kwargs={
    "language": LANGUAGE, 
    "task": "transcribe",
    "max_length": 448, # Note: don't exceed 448 - otherwise you'll get index errors when max_length exceeds the models positional encoding limits
    "num_beams": 1,
    "do_sample": False
}    
print(f"Using model: {pipe.model.name_or_path} with language: {LANGUAGE}")

# chose test_dataset or dev_dataset
dataset = dev_dataset
# dataset = test_dataset

N = len(dataset)
print(f"Number of examples in dataset: {N}")

results = []
for out in tqdm(pipe(AudioTextDataset(dataset, utterance_fields=['audio_id', 'speaker_id',  'language', 
                         'prompt_type', 'prompt_id',
                         'transcription', 'audio_length', 'transcript_length']), batch_size=32, generate_kwargs=generate_kwargs), total=N):
    results.append(out)

Using model: openai/whisper-tiny with language: sw
Number of examples in dataset: 272


  0%|          | 0/272 [00:00<?, ?it/s]

/usr/local/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


## Get overall results

In [16]:
# Finalizing results...

print("Finalizing results...")
results_df = prepare_results(results)
print("Getting speaker metadata...")
results_df = add_speaker_metadata(results_df, metadata_df)
print(f"Calculating WER and CER for {len(results_df)} examples...")
results_df = calculate_error_rates(results_df, verbose=False)
results_df.head(5)

Finalizing results...
Getting speaker metadata...
Calculating WER and CER for 272 examples...
Overall WER (normalized): 1
Overall CER (normalized): 1
Avg WER (normalized): 0.988
Avg CER (normalized): 0.716


,prompt_id,audio_id,prompt_type,transcript_length,language,speaker_id,audio_length,prediction,ground_truth,gender,age,type_nonstandard_speech,etiology,comments,slp_id,severity,ground_truth_normalized,prediction_normalized,wer,cer
0,SWA_002,KES004_SW002,text prompt,28,KSWA,KES004,25.752,kikilat di di semafamila iya tu un kutana kik...,Kila Disemba familia yetu hukutana kijijini kw...,Male,25-30,Stuttering (Disfluency Disorders),Neurological disorder,Swahili and English done by different Speech T...,KET001/KET002,severe,kila disemba familia yetu hukutana kijijini kw...,kikilat di di semafamila iya tu un kutana kik...,1.000000,0.447674
1,SWA_005,KES004_SW005,text prompt,37,KSWA,KES004,27.408,"Ketika shule sengi ali lelangats, dini lili j...",Katika shule ya msingi ya Langas ni ni ni nili...,Male,25-30,Stuttering (Disfluency Disorders),Neurological disorder,Swahili and English done by different Speech T...,KET001/KET002,severe,katika shule ya msingi ya langas ni ni ni nili...,ketika shule sengi ali lelangats dini lili ji...,1.000000,1.000000
2,SWA_006,KES004_SW006,text prompt,22,KSWA,KES004,23.352,"ili pofanyas ofahria kikikini, kikikikikikiki...",Nilipofanya safari ya kitalii katika mbuga ya ...,Male,25-30,Stuttering (Disfluency Disorders),Neurological disorder,Swahili and English done by different Speech T...,KET001/KET002,severe,nilipofanya safari ya kitalii katika mbuga ya ...,ili pofanyas ofahria kikikini kikikikikikikik...,1.000000,1.000000
3,SWA_008,KES004_SW008,text prompt,23,KSWA,KES004,28.344,kila juma pilim umumumumumumumumumumumumumumu...,Kila Jumapili mama huamka mapena na kutengenez...,Male,25-30,Stuttering (Disfluency Disorders),Neurological disorder,Swahili and English done by different Speech T...,KET001/KET002,severe,kila jumapili mama huamka mapena na kutengenez...,kila juma pilim umumumumumumumumumumumumumumu...,0.956522,1.000000
4,SWA_009,KES004_SW009,text prompt,18,KSWA,KES004,15.552,"nilipakua chou, nilipakua, nilipakua, nilipak...",Nilipokua chuo kikuu Nairobi tulikua na na uta...,Male,25-30,Stuttering (Disfluency Disorders),Neurological disorder,Swahili and English done by different Speech T...,KET001/KET002,severe,nilipokua chuo kikuu nairobi tulikua na na uta...,nilipakua chou nilipakua nilipakua nilipakua ...,1.000000,1.000000


## Get aggregated results

* we aggregate on the per speaker level first, and then can further aggregate by certain aspects

In [ ]:
per_speaker_agg = results_df[['speaker_id', 'severity', 'etiology', 'wer', 'cer']].groupby(['speaker_id', 'severity', 'etiology']).agg(['mean', 'count'])


In [ ]:
# per-severity results
per_speaker_agg.groupby(['severity']).agg(['mean', 'count']).round(2)


In [ ]:
# per-speaker results
per_speaker_agg.sort_values(by=['severity','speaker_id']).round(2)

In [ ]:
# per-etiology results
per_speaker_agg.groupby(['etiology']).agg(['mean', 'count']).round(2)

# Safe Predictions

In [ ]:
! mkdir /tmp/predictions

In [ ]:
LOCAL_OUTPUT_FOLDER = '/tmp/predictions'
if not os.path.exists(LOCAL_OUTPUT_FOLDER):
    os.makedirs(LOCAL_OUTPUT_FOLDER)
print('saving predictions to folder: ', LOCAL_OUTPUT_FOLDER)

output_filename = os.path.join(LOCAL_OUTPUT_FOLDER, DATASET_NAME.replace('cdli/', '') + '_' + WHISPER_MODEL_NAME.replace('openai/','').replace('cdli/', '') + '.tsv')
print('local output filename:', output_filename)
results_df.to_csv(output_filename, index=False, sep='\t', encoding='utf-8') 


# download locally
create_download_link(output_filename)
print("Click on this link to start download...")
